# 找到相关页面后补上相邻内容

LDA 推导题需要第 41～44 页。普通检索取 4 页时，只找到第 44 页，其余三页来自别处；但第一个结果正好是这段连续推导的最后一页。这里从第 44 页向前展开三页，再用相同的 4 页数量比较。

本节使用《南瓜书》原文和 BM25。补充内容发生在第一次检索之后，不改变原问题，也不增加返回页数。前后比较固定 4 个片段和同一字符上限，并只输出页码、字符数和回答要点。

In [1]:
import sys
from pathlib import Path


def find_course_root(start: Path) -> Path:
    for folder in (start, *start.parents):
        if (folder / "data" / "dataset/manifest.json").is_file():
            return folder
    raise FileNotFoundError("没有找到教程数据目录，请从本节所在目录运行。")


course_root = find_course_root(Path.cwd())
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))

from common.eval_utils import build_bm25_search, load_query_catalog, load_pdf_pages, page_coverage
from common.nontraining_utils import load_annotation

case = next(case for case in load_query_catalog() if case["id"] == "lda_recursive_derivation")
pages = load_pdf_pages()
page_by_number = {page["page"]: page for page in pages}
search = build_bm25_search(pages)

print("问题：", case["query"])
print("页面数：", len(pages))

问题： LDA 从投影分离目标怎样推到 N−1 个最大广义特征值及其特征向量？请给出中间优化关系。
页面数： 196


In [2]:
# 先完成原问题检索，再读取必要页清单做核对。
before = search(case["query"], top_k=4)
first_page = before[0].page
expanded_page_numbers = list(range(first_page - 3, first_page + 1))
after = [
    type(before[0])(number, page_by_number[number]["text"], 0.0)
    for number in expanded_page_numbers
]
annotation = load_annotation(case["id"])
expected_pages = annotation["expected_pages"]
before_coverage = page_coverage(expected_pages, before)[1]
after_coverage = page_coverage(expected_pages, after)[1]

print("普通检索的 4 页：", [item.page for item in before])
print("检索后用于核对的必要页：", expected_pages)
print("从第一条向前展开：", [item.page for item in after])
print("必要页面覆盖率：", before_coverage, "→", after_coverage)
assert len(before) == len(after) == 4 and before_coverage == 0.25 and after_coverage == 1.0

普通检索的 4 页： [44, 131, 126, 139]
检索后用于核对的必要页： [41, 42, 43, 44]
从第一条向前展开： [41, 42, 43, 44]
必要页面覆盖率： 0.25 → 1.0


## 看补回了什么

下面清理广告和控制字符，只核对四页中的回答要点，并报告统一字符上限下的实际正文长度。

In [3]:
import re

def clean_text(text):
    value = re.sub(r"→_→.*?←_←", "", str(text), flags=re.S)
    return re.sub(r"[\x00-\x1f\x7f-\x9f]", "", value).strip()


def limit_page_context(evidence, char_limit):
    limited = []
    remaining = char_limit
    for item in evidence:
        text = clean_text(item.text)[:max(remaining, 0)]
        limited.append(type(item)(item.page, text, item.score))
        remaining -= len(text)
    return limited


def context_chars(evidence):
    return sum(len(item.text) for item in evidence)

before_clean = [type(item)(item.page, clean_text(item.text), item.score) for item in before]
after_clean = [type(item)(item.page, clean_text(item.text), item.score) for item in after]
context_cap = min(context_chars(before_clean), context_chars(after_clean))
before_context = limit_page_context(before_clean, context_cap)
after_context = limit_page_context(after_clean, context_cap)

# 这是对回答要点的轻量复核，不把它当作完整的答案评分。
required_points = {
    "投影目标": ("同类样本", "很相近", "异类样本", "很疏远"),
    "散度比值": ("Sb", "Sw", "max W"),
    "拉格朗日关系": ("拉格朗日", "SbW", "SwW"),
    "特征值结论": ("N−1", "最大的λi", "广义特征值", "特征向量"),
}
def point_hits(evidence):
    text = re.sub(r"\s+", "", "".join(item.text for item in evidence))
    return {name: all(term.replace(" ", "") in text for term in terms) for name, terms in required_points.items()}

before_points = point_hits(before_context)
after_points = point_hits(after_context)
print("最终片段数：", len(before_context), "→", len(after_context), "；字符上限：", context_cap, "；实际字符数：", context_chars(before_context), "→", context_chars(after_context))
print("检索次数：1 → 1")
print("回答要点（普通检索 → 相邻页扩展）：")
for name in required_points:
    print(f"  {name}：{before_points[name]} → {after_points[name]}")
assert len(before_context) == len(after_context) == 4 and context_chars(before_context) == context_chars(after_context) == context_cap and sum(before_points.values()) < len(required_points) and all(after_points.values())

最终片段数： 4 → 4 ；字符上限： 4534 ；实际字符数： 4534 → 4534
检索次数：1 → 1
回答要点（普通检索 → 相邻页扩展）：
  投影目标：False → True
  散度比值：True → True
  拉格朗日关系：False → True
  特征值结论：True → True


## 结果和使用限制

普通检索返回 4 页，只覆盖必要页面的 25%；从第 44 页向前展开相同数量的页面后，第 41～44 页全部找回，覆盖率达到 100%。在清理广告和控制字符、统一字符上限后，轻量复核仍显示投影目标、散度关系、拉格朗日关系和特征值结论从部分缺失变为全部出现。这项提升来自文档中的连续位置，不是增加返回数量。

前提是第一条结果确实属于目标推导，而且缺失内容就在它前后。第一条结果本身不相关，或者资料并非按顺序展开时，不采用。

## 第二次检查：高斯混合聚类的相邻推导

再换一道高斯混合聚类题检查。普通检索先找到第 118 页的 EM 迭代说明，但式（9.35）的协方差推导在前面的第 116 页。从第一条结果向前取 4 页，和普通检索保持相同资料量。

In [4]:
gmm_adjacent_case = next(case for case in load_query_catalog() if case["id"] == "gmm_em_adjacent_derivation")
gmm_adjacent_before = search(gmm_adjacent_case["query"], top_k=4)
gmm_adjacent_first_page = gmm_adjacent_before[0].page
gmm_adjacent_page_numbers = list(range(gmm_adjacent_first_page - 3, gmm_adjacent_first_page + 1))
gmm_adjacent_after = [
    type(gmm_adjacent_before[0])(number, page_by_number[number]["text"], 0.0)
    for number in gmm_adjacent_page_numbers
]
gmm_adjacent_before_clean = [type(item)(item.page, clean_text(item.text), item.score) for item in gmm_adjacent_before]
gmm_adjacent_after_clean = [type(item)(item.page, clean_text(item.text), item.score) for item in gmm_adjacent_after]
gmm_adjacent_context_cap = min(context_chars(gmm_adjacent_before_clean), context_chars(gmm_adjacent_after_clean))
gmm_adjacent_before_context = limit_page_context(gmm_adjacent_before_clean, gmm_adjacent_context_cap)
gmm_adjacent_after_context = limit_page_context(gmm_adjacent_after_clean, gmm_adjacent_context_cap)
gmm_adjacent_annotation = load_annotation(gmm_adjacent_case["id"])
gmm_adjacent_expected = gmm_adjacent_annotation["expected_pages"]
gmm_adjacent_before_coverage = page_coverage(gmm_adjacent_expected, gmm_adjacent_before)[1]
gmm_adjacent_after_coverage = page_coverage(gmm_adjacent_expected, gmm_adjacent_after)[1]

def gmm_adjacent_points(evidence):
    text = re.sub(r"\s+", "", "".join(item.text for item in evidence))
    return {
        "式(9.35) 的协方差更新": all(term in text for term in ("公式(9.35)", "γji", "Σi")),
        "EM 更新并迭代到停止条件": all(term in text for term in ("EM算法", "更新", "反复迭代", "停止条件")),
    }

gmm_adjacent_before_points = gmm_adjacent_points(gmm_adjacent_before_context)
gmm_adjacent_after_points = gmm_adjacent_points(gmm_adjacent_after_context)
print("普通检索的 4 页：", [item.page for item in gmm_adjacent_before])
print("第一条结果页：", gmm_adjacent_first_page)
print("从第一条向前展开的 4 页：", gmm_adjacent_page_numbers)
print("必要页面覆盖：", gmm_adjacent_before_coverage, "→", gmm_adjacent_after_coverage)
print("必要回答要点（普通检索 → 相邻页扩展）：")
for name in gmm_adjacent_before_points:
    print(f"  {name}：{gmm_adjacent_before_points[name]} → {gmm_adjacent_after_points[name]}")
print("检索次数：1 → 1；最终资料量：", len(gmm_adjacent_before_context), "/", len(gmm_adjacent_after_context), "；字符上限：", gmm_adjacent_context_cap, "；实际字符数：", context_chars(gmm_adjacent_before_context), "→", context_chars(gmm_adjacent_after_context))
assert gmm_adjacent_first_page == 118 and gmm_adjacent_page_numbers == [115, 116, 117, 118]
assert len(gmm_adjacent_before_context) == len(gmm_adjacent_after_context) == 4 and context_chars(gmm_adjacent_before_context) == context_chars(gmm_adjacent_after_context) == gmm_adjacent_context_cap and gmm_adjacent_before_coverage == 0.5 and gmm_adjacent_after_coverage == 1.0

普通检索的 4 页： [118, 117, 38, 176]
第一条结果页： 118
从第一条向前展开的 4 页： [115, 116, 117, 118]
必要页面覆盖： 0.5 → 1.0
必要回答要点（普通检索 → 相邻页扩展）：
  式(9.35) 的协方差更新：False → True
  EM 更新并迭代到停止条件：True → True
检索次数：1 → 1；最终资料量： 4 / 4 ；字符上限： 4414 ；实际字符数： 4414 → 4414


相邻页扩展让式（9.35）的协方差更新从缺失变为出现，同时保留第 118 页的 EM 更新和停止条件；4 个片段、相同字符上限和一次检索的预算保持不变，必要页面覆盖从 50% 提高到 100%。



## 为什么相邻页扩展有时有效

按页扩展是 Small-to-Big 的一种文档级变体：先用检索器找到一个可信的中心页，再在原文顺序中取它前后的页。它只适合“第一条确实属于目标推导，而且缺口就在前后”的情况；中心页不相关、章节顺序不稳定或答案需要跨章节时，不应默认扩展。

使用时固定返回页数和字符上限，并记录被替换掉的页。本页先看“LDA 从投影分离目标怎样推到 N−1 个最大广义特征值及其特征向量？”，再看“公式(9.35) 中的协方差如何在 EM 迭代中更新，并在何时停止？”；两道题都用同样的片段数和字符预算作前后比较。


In [5]:
# 实现要点：相邻页扩展应保留中心页，并显式限制边界。
def expand_adjacent_pages(first_page, page_text, radius=3, limit=4):
    candidates = [p for p in range(first_page - radius, first_page + radius + 1) if p in page_text]
    ranked = sorted(candidates, key=lambda p: (abs(p - first_page), p))
    selected = {first_page}
    for page in ranked:
        if len(selected) >= limit: break
        selected.add(page)
    return sorted(selected)

def ordered_context(page_numbers, page_text, char_budget):
    out, remaining = [], char_budget
    for page in sorted(page_numbers):
        text = str(page_text[page])[:max(0, remaining)]
        out.append(text); remaining -= len(text)
    return "\n\n".join(out)

# 这里的页码只在检索完成后用于对照，不能提前喂给检索器。


## 相邻页扩展的机制、适用条件与代码要点

相邻页扩展是 Small-to-Big 的文档级变体：先用原问题找到一个可信的中心页，再在原文顺序中取中心页前后的页。流程可以写成“检索中心页 → 检查中心页是否属于目标推导 → 在合法页码范围内补邻页 → 固定返回页数和字符预算 → 核对必要要点”。它不改变原问题，也不通过增加返回数量伪装成更高召回。

在 LDA 推导问题上，普通返回只覆盖必要页的一部分，从第 44 页向前扩展后找回第 41～44 页；在 GMM 协方差更新问题上，从第 118 页向前扩展后补回第 116 页的更新说明，同时保留第 118 页的 EM 停止条件。两组对照都固定 4 个片段和同一字符上限，比较的是上下文组织方式，不是多返回资料。

这项方法只在三个条件同时满足时值得使用：中心页确实与问题相关；缺口位于中心页的前后连续范围；资料的页序有稳定语义。中心页不相关、答案跨章节，或 PDF 页码顺序与论证顺序不一致时，应改写问题或扩大检索范围，而不是机械展开。实现里的 `range(first_page - radius, first_page + radius + 1)` 必须检查页码存在性，并记录被替换的原页，避免越界，也方便复查结果。

这页的“相邻页扩展”用于解释文档级父子关系，不能和前一页 Small-to-Big/AutoMerging 的父子片段实验重复计为新的独立方法收益。保存输出只说明上述两个当前案例；不外推成所有教材或所有 PDF 都适合按页扩展。

In [6]:
from common.eval_utils import emit_tutorial_audit

# 统一保存契约：页码来自实际检索/扩展结果，标注只在流水线完成后用于指标。
import json

def _actual_pages(items):
    pages = []
    for item in items:
        values = item.pages if hasattr(item, 'pages') else [item.page]
        for page in values:
            page = int(page)
            if page not in pages:
                pages.append(page)
    return pages

def _metrics(items, expected_pages):
    pages = _actual_pages(items)
    expected = {int(page) for page in expected_pages}
    found = set(pages) & expected
    rank = next((index for index, page in enumerate(pages, 1) if page in expected), None)
    return {'pages': pages, 'first_required_rank': rank,
            'required_page_coverage': len(found) / len(expected) if expected else 0.0}

def _emit(method, role, case_id, before_items, after_items, purpose=None):
    annotation = load_annotation(case_id)
    payload = {'case_id': case_id, 'method': method, 'role': role,
              'before': _metrics(before_items, annotation['expected_pages']),
              'after': _metrics(after_items, annotation['expected_pages'])}
    if purpose:
        payload['check_purpose'] = purpose
    emit_tutorial_audit(payload)

_emit('展开相邻片段', 'main', 'lda_recursive_derivation', before, after)
_emit('展开相邻片段', 'check', 'gmm_em_adjacent_derivation',
      gmm_adjacent_before, gmm_adjacent_after, '再次改善')


## 本页导航

- 章节入口：[本章 README](README.md)
- 运行准备：[C7 统一运行准备](../README.md#运行准备)
- 相关下一步：[按句子补充上下文](按句子和父子片段补充上下文.ipynb)

